In [0]:
from pyspark.sql.functions import input_file_name, current_timestamp, current_date

In [0]:
dbutils.widgets.text("catalog_name", "dbr_dev")
dbutils.widgets.text("container", "trezio2005")
dbutils.widgets.text("storage_account", "dlspl21databricks")

catalog_name = dbutils.widgets.get("catalog_name")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")


In [0]:
source_path = f"/Volumes/{catalog_name}/trezio2005_bronze/raw_data/"
checkpoint_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/checkpoints/bronze"
target_table = f"{catalog_name}.trezio2005_bronze.ingested_data"
schema_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/schemas/bronze"

In [0]:
#Autolader
df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", schema_path)
      .option("cloudFiles.inferColumnTypes", "true") 
      .option("header", "true") 
      .load(source_path)
      .selectExpr("*", "_metadata.file_name as source_filename")
      .withColumn("ingestion_timestamp", current_timestamp())
      .withColumn("load_date", current_date())
)
df = df.toDF(*[col.replace(' ', '_') for col in df.columns])

#Write to Bronze layer delta table
(df.writeStream
   .format("delta")
   .option("checkpointLocation", checkpoint_path) 
   .trigger(availableNow=True) 
   .toTable(target_table)
)

In [0]:
#count = spark.read.table(target_table).count()
#print(f"Total records ingested: {count:,}")